# 1. Data exploration — MulTaBench core text

A full audit of the **20 official core TEXT datasets**, without modelling or API calls. Definitions, data versions and SHA-256 hashes are pinned in `config/exploration/datasets.yaml`. The original template example remains separate.

## 1. Scope and provenance

The public Kaggle CSVs and metadata are loaded at their pinned versions. With `allow_download: true`, a missing dataset is downloaded; set it to false for offline work. No upstream model code is installed or run. Each notebook can run independently. See `docs/DATA_SOURCES.md` for the official text rule and its limitations.

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import load_config, save_resolved_config
from src.utils.notebook_display import display_local, display_json
from src.utils.provenance import provenance
from src.data.audit import audit_collection
from src.visualize import figures, style
style.apply()
cfg = load_config()
from src.data.audit import audit_summary
from src.visualize.audit import burden_figures
import matplotlib.pyplot as plt

NOTEBOOK = "01_data_exploration"
from src.utils.notebook_report import begin_report, report_section, finish_report
begin_report()
report_section('1. Scope and provenance')
save = figures.FigureSaver(NOTEBOOK)
config_hash = save_resolved_config(cfg, NOTEBOOK)
display_json("Execution provenance", provenance())
audit = audit_collection(cfg)


## 2. Dataset overview

One row per dataset. Feature counts exclude the target. Binary versus multiclass is determined from the official classification task and observed distinct class vocabulary; numerical targets are not automatically treated as regression. Class vocabulary may be inspected below; no frequencies enter request metadata.

In [ ]:
report_section('2. Dataset overview')
display_local(audit.summary[["dataset", "rows", "features", "task_type", "target", "target_dtype", "n_classes", "text_columns", "non_text_columns"]])
for dataset_id in audit.summary.dataset_id:
    display_json(audit.metadata(dataset_id).dataset_name + " — task metadata", audit.details[dataset_id]["task_metadata"])


## 3. Column, text and target audit

Expand a dataset to see **every feature**, missingness, unique counts/ratios, numeric statistics, categorical cardinalities/common values, text-length statistics and representative examples. Unique ratios use nonmissing rows as denominator. Character/word/token length statistics include missing text as empty, matching workload accounting. “Words” means whitespace-separated strings.

Tokens are approximated as `ceil(characters / 4)`, not measured with Jev’s tokenizer. Examples are unique nonempty values at deterministic length ranks near 10%, 50% and 90%, clipped only for display. Full request text is never clipped.

The target distribution or regression quantiles are **audit-only**. The request builder has no input for these statistics. “Other” includes date-like columns. Non-text feature classification uses the CSV dtype; no model preprocessing has been selected.

In [ ]:
report_section('3. Column, text and target audit')
for dataset_id in audit.summary.dataset_id:
    detail = audit.details[dataset_id]
    display_json(audit.metadata(dataset_id).dataset_name + " — full column audit", {
        "features": audit.columns[audit.columns.dataset_id == dataset_id].to_dict(orient="records"),
        "text_statistics": audit.text[audit.text.dataset_id == dataset_id].to_dict(orient="records"),
        "numeric_statistics": audit.numeric[audit.numeric.dataset_id == dataset_id].fillna("missing").to_dict(orient="records"),
        "categorical_statistics": audit.categorical[audit.categorical.dataset_id == dataset_id].to_dict(orient="records"),
        "text_examples": detail["text_examples"],
        "target_audit_only": detail["target_audit_only"],
        "issues": detail["issues"],
    })


## 4. Text burden

Combined row tokens use `ceil(sum(text characters) / 4)`. Notebook 2 adds metadata, question wording, class choices and JSON escaping. No non-text features enter Jev. Short strings and names count as text under the official annotation.


In [ ]:
report_section('4. Text burden')
display_local(audit.summary[["dataset", "text_columns", "combined_tokens_mean", "combined_tokens_median", "combined_tokens_p95", "combined_tokens_max", "total_text_tokens"]])
for fig, name, caption in burden_figures(audit.summary, cfg["token_estimation"]["characters_per_token"]):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


## 5. Saved outputs

CSV/JSON audits and encoded length arrays are under `output_JEVPFN/exploration/results/data_audit/`. Configurations are in that phase's `manifests/`. Figures are in `figures/01_data_exploration/`, with `captions.md`. The full final summary contains every displayed audit detail and plotted value, including local examples. Generated reports are gitignored; source notebooks remain output-free for GitHub.


In [ ]:
report_section('5. Saved outputs')
display_json("Audit artifacts", {"directory": str(audit.folder), "tables": ["summary.csv", "columns.csv", "text.csv", "numeric.csv", "categorical.csv"], "config_sha256": config_hash})


## Summary

Section order matches the notebook. The notebook saves this complete text verbatim into `output_JEVPFN/allresults.md`.

In [ ]:
finish_report(NOTEBOOK, audit_summary(audit) + "\n" + save.summary())
